# CMMD2022 Veri Hazirlik — Parca 2

`PLAN.md` Bolum 6'daki NYU pipeline'ini uygular: meme etrafini kirp, **sag memeleri cevir**,
best-center penceresi, goruntu basina z-score. Ayrica hasta-seviyesi stratified split ve sabit
degerlendirme vakalarini uretir.

Girdi: ham DICOM + `CMMD_clinicaldata_revision.xlsx` + `ozet/haric_hastalar.json` (Parca 1)

Cikti:
- `hazir/<hasta>_<taraf>_<view>.png` — kirpilmis + yon normalize edilmis goruntu (kayipsiz)
- `hazir/manifest.csv` — her goruntu icin boyut, best_center, etiket, split
- `hazir/split.json` — hasta seviyesinde train/val/test
- `hazir/eval_cases.json` — tum deneylerde ortak 6 sabit test vakasi
- `hazir/config.json` — `GLOBAL_SIZE` ve on isleme sabitleri

**Neden pencere degil kirpilmis goruntu onbellege alaniyor:** pencere yerlesimi egitim sirasinda
gurultuyle degisiyor (tek augmentasyon bu). Onbellege son pencereyi yazmak augmentasyonu
oldururdu. z-score da egitimde uygulanir, pencere secildikten sonra.

In [ ]:
import os, glob, json, hashlib
from pathlib import Path

ORTAM = 'colab' if os.path.isdir('/content') else 'yerel'
print('ortam:', ORTAM)
if ORTAM == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')

ADAY_KOKLER = [
    r'D:\mamografi\multiple_instance_classifier\archive\TheChineseMammographyDatabase',
    '/content/drive/MyDrive/multiple_instance_classifier/archive/TheChineseMammographyDatabase',
    '/content/drive/MyDrive/multiple_instance_classifier/archive',
]
DATA_ROOT = next((p for p in ADAY_KOKLER if os.path.isdir(p)), None)

KOK = Path('/content/drive/MyDrive/multiple_instance_classifier' if ORTAM == 'colab' else Path.cwd())
OZET_DIR = KOK / 'ozet'
HAZIR_DIR = KOK / 'hazir'
HAZIR_DIR.mkdir(parents=True, exist_ok=True)
print('DATA_ROOT :', DATA_ROOT)
print('HAZIR_DIR :', HAZIR_DIR)

In [ ]:
try:
    import pydicom, cv2
except ImportError:
    !pip -q install pydicom opencv-python-headless
    import pydicom, cv2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

SEED = 42
ESIK = 10          # meme maskesi esigi (Bolum 6'da olculdu, asagida yeniden dogrulaniyor)
MAX_CROP_NOISE = (100, 100)   # NYU: random_augmentation_best_center
MAX_CROP_SIZE_NOISE = 100
pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 60)

## 1. Goruntu tablosu + etiket birlestirme + dislamalar

Parca 1'in bulgularini uygular: view `ViewCodeSequence`'den, `haric_hastalar.json`'daki 4 hasta
cikarilir, klinik etiketi olmayan memeler dusurulur.

In [ ]:
CC_MLO = {'cranio-caudal': 'CC', 'medio-lateral oblique': 'MLO'}
dicomlar = sorted(glob.glob(os.path.join(DATA_ROOT, '**', '*.dcm'), recursive=True))
print('dicom:', len(dicomlar))

kayit = []
for i, p in enumerate(dicomlar):
    d = pydicom.dcmread(p, stop_before_pixels=True, force=True)
    vcs = getattr(d, 'ViewCodeSequence', None)
    kod = str(getattr(vcs[0], 'CodeMeaning', '')) if vcs else ''
    kayit.append(dict(yol=p, hasta=str(d.PatientID), taraf=str(d.ImageLaterality),
                      view=CC_MLO.get(kod), bits=int(getattr(d, 'BitsStored', 8))))
    if (i + 1) % 1000 == 0:
        print(f'  {i+1}/{len(dicomlar)}')
g = pd.DataFrame(kayit)
g['meme'] = g.hasta + '_' + g.taraf
assert g.view.notna().all(), 'view cozulemedi'

klinik = pd.read_excel(os.path.join(DATA_ROOT, 'CMMD_clinicaldata_revision.xlsx'))
klinik['meme'] = klinik.ID1.astype(str) + '_' + klinik.LeftRight.astype(str)

with open(OZET_DIR / 'haric_hastalar.json', encoding='utf-8') as f:
    haric = json.load(f)['hastalar']
print('haric hasta (Parca 1):', haric)

n0 = len(g)
g = g[~g.hasta.isin(haric)]
print(f'dislama sonrasi goruntu: {n0} -> {len(g)}')

tablo = g.merge(klinik[['meme', 'Age', 'abnormality', 'classification', 'subtype']],
                on='meme', how='inner')
print(f'etiketli goruntu: {len(tablo)}  meme: {tablo.meme.nunique()}  hasta: {tablo.hasta.nunique()}')
tablo['grup'] = tablo.hasta.str[:2]
tablo['y'] = (tablo.classification == 'Malignant').astype(int)
print(tablo.classification.value_counts().to_string())
tablo.head()

## 2. On isleme fonksiyonlari — `hazir/onisleme.py`

Bu fonksiyonlar egitim notebook'unda da gerekiyor (pencere yerlesimi ve z-score calisma aninda
uygulaniyor). Iki yerde ayri kopya tutmak, kopyalarin sessizce birbirinden kaymasi riskini
tasiyor -- egitim farkli normalizasyonla kosar ve kimse fark etmez. Bu yuzden **tek kaynak bu
hucre**: modul dosyasi buradan yazilir, hem bu notebook hem egitim notebook'u onu import eder.

NYU pipeline karsiliklari: `crop_mammogram.py` -> `kirp_ve_cevir`,
`random_augmentation_best_center` -> `pencere_al`, `standard_normalize_single_image` -> `z_score`.

In [ ]:
ONISLEME_KOD = '''# Parca 2 tarafindan uretildi -- tek kaynak gmic_cmmd_veri_hazirlik.ipynb.
import numpy as np
import cv2

ESIK = {esik}


def en_buyuk_bilesen(maske):
    \"\"\"En buyuk bagli bileseni ve sinirlayici kutusunu dondurur.\"\"\"
    n, lab, st, _ = cv2.connectedComponentsWithStats(maske.astype(np.uint8), 8)
    if n <= 1:
        return maske, (0, 0, maske.shape[1], maske.shape[0])
    i = 1 + int(np.argmax(st[1:, cv2.CC_STAT_AREA]))
    x, y, w, h = st[i, :4]
    return lab == i, (int(x), int(y), int(w), int(h))


def kirp_ve_cevir(im, taraf, esik=ESIK):
    \"\"\"Meme etrafini kirp, sag memeyi yatay cevir (tum goruntuler saga bakar).\"\"\"
    cc, (x, y, w, h) = en_buyuk_bilesen(im > esik)
    kirpik = im[y:y + h, x:x + w]
    maske = cc[y:y + h, x:x + w]
    if taraf == 'R':
        kirpik, maske = kirpik[:, ::-1].copy(), maske[:, ::-1].copy()
    return kirpik, maske


def best_center(maske):
    \"\"\"Pencere merkezi: meme maskesinin kutle merkezi.\"\"\"
    sy, sx = maske.sum(1), maske.sum(0)
    cy = float(np.average(np.arange(len(sy)), weights=sy)) if sy.sum() else len(sy) / 2
    cx = float(np.average(np.arange(len(sx)), weights=sx)) if sx.sum() else len(sx) / 2
    return cy, cx


def pencere_al(im, merkez, boyut, gurultu=(0, 0), boyut_gurultu=0, rng=None):
    \"\"\"boyut=(H,W) pencereyi merkeze oturtur, tasani iceri kaydirir, eksigi 0 ile doldurur.

    Egitimde gurultu ile cagrilir (NYU'nun tek augmentasyonu). Boyut gurultusunun tam semantigi
    yayinlanan koddan cikarilamadigi icin sunun gibi uygulaniyor: pencere +-boyut_gurultu
    buyuklukte alinip hedef boyuta yeniden olceklenir (yaklasik %4 zoom jitter).
    \"\"\"
    H, W = boyut
    if rng is not None and boyut_gurultu:
        d = int(rng.uniform(-1, 1) * boyut_gurultu)
        H, W = H + d, W + d
    cy, cx = merkez
    if rng is not None and any(gurultu):
        cy += rng.uniform(-1, 1) * gurultu[0]
        cx += rng.uniform(-1, 1) * gurultu[1]

    y0 = int(round(cy - H / 2)); x0 = int(round(cx - W / 2))
    y0 = max(0, min(y0, max(im.shape[0] - H, 0)))   # shift_window_inside_image
    x0 = max(0, min(x0, max(im.shape[1] - W, 0)))
    dilim = im[y0:y0 + H, x0:x0 + W]

    if dilim.shape != (H, W):   # kirpik goruntu pencereden kucukse doldur
        pad = np.zeros((H, W), dtype=im.dtype)
        pad[:dilim.shape[0], :dilim.shape[1]] = dilim
        dilim = pad
    if (H, W) != tuple(boyut):
        dilim = cv2.resize(dilim, (boyut[1], boyut[0]), interpolation=cv2.INTER_AREA)
    return dilim


def z_score(im, esik=None):
    \"\"\"Goruntu basina (x-mean)/max(std,1e-5) -- NYU standard_normalize_single_image.

    Istatistikler yalnizca meme dokusu uzerinden aliniyor (esik verilirse). Gerekce Bolum 5b'de
    olculdu: pencerenin alan olarak ~yarisi padding ve padding orani goruntuden goruntuye
    degistigi icin pencere geneli istatistik normalizasyonu doluluk oranina baglar.
    \"\"\"
    x = im.astype(np.float32)
    m = (im > esik) if esik is not None else np.ones_like(im, bool)
    if m.sum() < 100:
        m = np.ones_like(im, bool)
    return (x - x[m].mean()) / max(x[m].std(), 1e-5)
'''.format(esik=ESIK)

(HAZIR_DIR / 'onisleme.py').write_text(ONISLEME_KOD, encoding='utf-8')

import importlib, sys
sys.path.insert(0, str(HAZIR_DIR))
import onisleme
importlib.reload(onisleme)
from onisleme import en_buyuk_bilesen, kirp_ve_cevir, best_center, pencere_al, z_score

print('yazildi ve import edildi:', HAZIR_DIR / 'onisleme.py')
print('fonksiyonlar:', [f.__name__ for f in
                        [en_buyuk_bilesen, kirp_ve_cevir, best_center, pencere_al, z_score]])

# hizli dogrulama: doku degisken olmali (sabit dolgu std=0 verir, z-score tanimsiz kalir)
_rng = np.random.default_rng(0)
_t = np.zeros((400, 300), np.uint8)
_t[50:350, 40:260] = _rng.integers(20, 250, (300, 220), dtype=np.uint16).astype(np.uint8)
_p = pencere_al(_t, best_center(_t > ESIK), (256, 256))
_z = z_score(_p, ESIK)
assert _p.shape == (256, 256)
assert abs(_z[_p > ESIK].mean()) < 1e-4 and abs(_z[_p > ESIK].std() - 1) < 1e-3
print('pencere ve maske ici z-score dogrulandi (doku ort ~0, std ~1)')

## 3. Esik secimi — olcumle

Sabit esik ile Otsu karsilastirilir. Olcut: **en buyuk bagli bilesenin disinda kalan doku orani**
(kirpmada kaybedilen meme dokusu). Otsu mamografide dusuk yogunluklu cevre dokuyu kesiyor — tam
lezyon marjlarinin bulundugu yer.

In [ ]:
def oku(yol):
    im = pydicom.dcmread(yol, force=True).pixel_array
    if im.dtype != np.uint8:
        # 5202 goruntunun 2'si 16-bit; geri kalanla ayni olceye indiriliyor
        im = (im.astype(np.float32) / max(im.max(), 1) * 255).astype(np.uint8)
    return im


rng = np.random.default_rng(SEED)
ornek = rng.choice(len(tablo), 30, replace=False)
kar = []
for i in ornek:
    im = oku(tablo.iloc[i].yol)
    doku = im > ESIK
    for ad, maske in [('sabit', doku),
                      ('otsu', im > cv2.threshold(im, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)[0])]:
        cc, (x, y, w, h) = en_buyuk_bilesen(maske)
        kar.append(dict(yontem=ad, w=w, h=h, alan_orani=w * h / im.size,
                        kayip_doku=float((doku & ~cc).sum() / max(doku.sum(), 1))))
kar = pd.DataFrame(kar)
print(kar.groupby('yontem')[['w', 'h', 'alan_orani', 'kayip_doku']].mean().round(4).to_string())
print()
print(kar.groupby('yontem').kayip_doku.describe()[['mean', '50%', 'max']].round(4).to_string())
print(f"\nKARAR: sabit esik {ESIK}. Otsu ortalama %{kar[kar.yontem=='otsu'].kayip_doku.mean()*100:.1f} "
      f"doku kaybediyor (en kotu %{kar[kar.yontem=='otsu'].kayip_doku.max()*100:.0f}), "
      f"sabit esik %{kar[kar.yontem=='sabit'].kayip_doku.mean()*100:.2f}.")

## 4. Gorsel dogrulama

Ham -> maske -> kirpilmis + yon normalize -> pencere. Dort ornekte ust uste bakilir; hepsinde
meme **saga** bakmali.

In [ ]:
ILK_TAHMIN = (2304, 1280)   # gorsel kontrol icin; kesin GLOBAL_SIZE Bolum 6'da manifest'ten
gorsel = [tablo[(tablo.taraf == t) & (tablo.view == v)].iloc[0]
          for t in ['L', 'R'] for v in ['CC', 'MLO']]

fig, ax = plt.subplots(len(gorsel), 4, figsize=(13, 3.2 * len(gorsel)))
for r, s in enumerate(gorsel):
    im = oku(s.yol)
    cc, _ = en_buyuk_bilesen(im > ESIK)
    kirpik, maske = kirp_ve_cevir(im, s.taraf)
    pen = pencere_al(kirpik, best_center(maske), ILK_TAHMIN)
    for c, (g, b) in enumerate([(im, f'{s.hasta} {s.taraf}-{s.view} ham {im.shape}'),
                                (cc, 'meme maskesi'),
                                (kirpik, f'kirpilmis+cevrilmis {kirpik.shape}'),
                                (pen, f'pencere (ilk tahmin) {pen.shape}')]):
        ax[r, c].imshow(g, cmap='gray'); ax[r, c].set_title(b, fontsize=8); ax[r, c].axis('off')
plt.tight_layout()
plt.savefig(OZET_DIR / 'onisleme_dogrulama.png', dpi=120, bbox_inches='tight')
plt.show()

## 5. Onbellek uretimi (tam gecis)

3.734 goruntu: oku -> kirp -> cevir -> kayipsiz PNG. `manifest.csv` boyut ve best_center'i tutar.
Tekrar kosuldugunda var olan dosyalar atlanir.

In [ ]:
YENIDEN = False   # True: var olan PNG'ler de yeniden uretilir

satir = []
for i, s in enumerate(tablo.itertuples(index=False)):
    ad = f'{s.hasta}_{s.taraf}_{s.view}.png'
    hedef = HAZIR_DIR / ad
    if hedef.exists() and not YENIDEN:
        kirpik = cv2.imread(str(hedef), cv2.IMREAD_GRAYSCALE)
        maske = kirpik > ESIK
    else:
        kirpik, maske = kirp_ve_cevir(oku(s.yol), s.taraf)
        gecici = hedef.with_suffix('.tmp.png')
        cv2.imwrite(str(gecici), kirpik)      # atomik yaz: yarim dosya okunmasin
        os.replace(gecici, hedef)
    cy, cx = best_center(maske)
    satir.append(dict(dosya=ad, hasta=s.hasta, taraf=s.taraf, view=s.view, meme=s.meme,
                      h=kirpik.shape[0], w=kirpik.shape[1], cy=round(cy, 1), cx=round(cx, 1),
                      y=s.y, classification=s.classification, abnormality=s.abnormality,
                      Age=s.Age, subtype=s.subtype, grup=s.grup))
    if (i + 1) % 500 == 0:
        print(f'  {i+1}/{len(tablo)}')

manifest = pd.DataFrame(satir)
print('\nonbellek:', len(manifest), 'goruntu')
print('kirpilmis boyut (h x w) ozeti:')
print(manifest[['h', 'w']].describe().round(0).to_string())

## 5b. Padding orani ve z-score — olcum

`GLOBAL_SIZE` p99'dan seciliyor, dolayisiyla goruntulerin neredeyse tamami pencereden kucuk ve
sifirla dolduruluyor. Padding orani goruntuden goruntuye degisiyor; z-score istatistigi pencere
genelinden alinirsa normalizasyon **dokuya degil doluluk oranina** baglanir. Asagida olculuyor.

In [ ]:
orn2 = manifest.sample(min(150, len(manifest)), random_state=SEED)
sat2 = []
for s in orn2.itertuples(index=False):
    im = cv2.imread(str(HAZIR_DIR / s.dosya), cv2.IMREAD_GRAYSCALE)
    pen = pencere_al(im, (s.cy, s.cx), ILK_TAHMIN)
    m = pen > ESIK
    sat2.append(dict(doluluk=m.mean(), tum_ort=pen.mean(), maske_ort=pen[m].mean()))
o = pd.DataFrame(sat2)

k_tum = np.corrcoef(o.doluluk, o.tum_ort)[0, 1]
k_maske = np.corrcoef(o.doluluk, o.maske_ort)[0, 1]
print(f'doku doluluk orani: ort {o.doluluk.mean():.2f}  ({o.doluluk.min():.2f} - {o.doluluk.max():.2f})')
print(f'pencere geneli ortalama <-> doluluk korelasyonu : {k_tum:+.3f}')
print(f'maske ici ortalama    <-> doluluk korelasyonu : {k_maske:+.3f}')
print()
print('KARAR: z-score istatistikleri maske ici (piksel > esik) alinir. Pencere geneli')
print('istatistik normalizasyonu padding oranina baglar; ayni goruntu iki farkli pencere')
print('boyutunda z-score lanirsa doku ortalamasi 2 kat kayiyor.')
print('NYU koduyla fark: onlarin penceresi kirpma boyutuna yakin oldugu icin padding kucuk;')
print('bizde alan olarak ~%46 padding var, bu yuzden bu sapma zorunlu.')

## 6. `GLOBAL_SIZE` karari

Pencere, kirpilmis goruntulerin buyuk kismini kaybetmeden ortmeli. Yuzdelikler uzerinden secilir
ve aga uygun olsun diye **32'nin kati**na yuvarlanir.

In [ ]:
for q in [50, 90, 95, 99, 100]:
    print(f'  p{q}: h={np.percentile(manifest.h, q):.0f}  w={np.percentile(manifest.w, q):.0f}')

def otuziki(v):
    return int(np.ceil(v / 32) * 32)

GLOBAL_H = otuziki(np.percentile(manifest.h, 99))
GLOBAL_W = otuziki(np.percentile(manifest.w, 99))
GLOBAL_SIZE = (GLOBAL_H, GLOBAL_W)
print(f'\nGLOBAL_SIZE = {GLOBAL_SIZE}  (p99, 32 katina yuvarlandi)')

tasan = ((manifest.h > GLOBAL_H) | (manifest.w > GLOBAL_W)).sum()
print(f'penceresine sigmayan goruntu: {tasan} / {len(manifest)} '
      f'(%{100*tasan/len(manifest):.1f}) -- bunlarda kenar kirpilir')
kucuk = ((manifest.h < GLOBAL_H) & (manifest.w < GLOBAL_W)).sum()
print(f'pencereden kucuk (0 ile doldurulacak): {kucuk} / {len(manifest)}')

config = dict(GLOBAL_SIZE=list(GLOBAL_SIZE), PATCH_SIZE=256, K=6, ESIK=ESIK, SEED=SEED,
              MAX_CROP_NOISE=list(MAX_CROP_NOISE), MAX_CROP_SIZE_NOISE=MAX_CROP_SIZE_NOISE,
              normalizasyon='goruntu basina z-score, istatistikler maske ici (piksel > ESIK)',
              yon='sag memeler yatay cevrildi, tum goruntuler saga bakar')
with open(HAZIR_DIR / 'config.json', 'w', encoding='utf-8') as f:
    json.dump(config, f, indent=2, ensure_ascii=False)
print('\nyazildi: config.json')

## 7. Hasta seviyesi stratified split

Katman: **sinif x alt kume**. Iki memesi farkli sinifta olan hastalar tek bir `karisik`
katmaninda (Parca 1: `karisik_D1` 17 + `karisik_D2` 12 hasta, ayri tutulursa %15'i 2 hastaya
duser). Ayni hastanin tum goruntuleri tek bolunmede.

In [ ]:
hasta = manifest.groupby('hasta').agg(
    sinif=('classification', lambda s: s.iloc[0] if s.nunique() == 1 else 'karisik'),
    grup=('grup', 'first')).reset_index()
hasta['katman'] = np.where(hasta.sinif == 'karisik', 'karisik', hasta.sinif + '_' + hasta.grup)
print('katmanlar:'); print(hasta.katman.value_counts().to_string())

egt, kalan = train_test_split(hasta, test_size=0.30, random_state=SEED, stratify=hasta.katman)
dgr, tst = train_test_split(kalan, test_size=0.50, random_state=SEED, stratify=kalan.katman)
split = {'train': sorted(egt.hasta), 'val': sorted(dgr.hasta), 'test': sorted(tst.hasta)}

# sizinti kontrolu
assert not (set(split['train']) & set(split['val'])), 'train/val hasta kesisimi'
assert not (set(split['train']) & set(split['test'])), 'train/test hasta kesisimi'
assert not (set(split['val']) & set(split['test'])), 'val/test hasta kesisimi'
assert sum(map(len, split.values())) == len(hasta), 'hasta sayisi tutmuyor'

hasta_split = {h: k for k, hs in split.items() for h in hs}
manifest['split'] = manifest.hasta.map(hasta_split)

print('\nhasta:', {k: len(v) for k, v in split.items()})
print('\ngoruntu x sinif:')
print(pd.crosstab(manifest.split, manifest.classification, margins=True).to_string())
print('\nmeme sayisi:')
print(manifest.groupby('split').meme.nunique().to_string())
print('\nmalign orani (meme, alt kume kirilimi):')
meme_tab = manifest.drop_duplicates('meme')
print(pd.crosstab(meme_tab.split, [meme_tab.grup, meme_tab.classification]).to_string())

manifest.to_csv(HAZIR_DIR / 'manifest.csv', index=False)
with open(HAZIR_DIR / 'split.json', 'w', encoding='utf-8') as f:
    json.dump(split, f, indent=2)
print('\nyazildi: manifest.csv, split.json')

## 8. Sabit degerlendirme vakalari

Test setinden 6 meme: `sinif x abnormality` cesitliligi gozetilerek, sabit seed. Tum deneylerde
ayni vakalar kullanilir (saliency map gorselleri karsilastirilabilir olsun).

In [ ]:
test_memeler = manifest[manifest.split == 'test'].drop_duplicates('meme')
istenen = [('Malignant', 'mass'), ('Malignant', 'calcification'), ('Malignant', 'both'),
           ('Benign', 'mass'), ('Benign', 'calcification'), ('Benign', 'both')]

secim = []
for sinif, anorm in istenen:
    aday = test_memeler[(test_memeler.classification == sinif) &
                        (test_memeler.abnormality == anorm)]
    if len(aday):
        secim.append(aday.sample(1, random_state=SEED).iloc[0])
    else:
        print(f'UYARI: test setinde {sinif}/{anorm} yok, atlandi')

eval_cases = [dict(meme=s.meme, hasta=s.hasta, taraf=s.taraf,
                   classification=s.classification, abnormality=s.abnormality,
                   Age=int(s.Age), subtype=(None if pd.isna(s.subtype) else s.subtype),
                   grup=s.grup) for s in secim]
with open(HAZIR_DIR / 'eval_cases.json', 'w', encoding='utf-8') as f:
    json.dump(eval_cases, f, indent=2, ensure_ascii=False)
print(pd.DataFrame(eval_cases).to_string(index=False))
print('\nyazildi: eval_cases.json')

## 9. Dogrulama

Onbellekten geri okuyup egitim yolunu birebir taklit eder: pencere (gurultulu) + z-score.
Beklenen: z-score sonrasi ortalama ~0, std ~1; meme saga bakar; gurultu pencereyi kaydirir.

In [ ]:
rng = np.random.default_rng(SEED)
kontrol = manifest.sample(6, random_state=SEED)

print('z-score dogrulamasi (pencere sonrasi):')
for s in kontrol.itertuples(index=False):
    im = cv2.imread(str(HAZIR_DIR / s.dosya), cv2.IMREAD_GRAYSCALE)
    pen = pencere_al(im, (s.cy, s.cx), GLOBAL_SIZE)
    z = z_score(pen, ESIK)
    print(f'  {s.dosya:28s} kirpik {im.shape} -> pencere {pen.shape} '
          f'doku ort {z[pen>ESIK].mean():+.4f} std {z[pen>ESIK].std():.4f} '
          f'| pencere geneli ort {z.mean():+.3f} min {z.min():+.2f} max {z.max():+.2f}')

s = kontrol.iloc[0]
im = cv2.imread(str(HAZIR_DIR / s.dosya), cv2.IMREAD_GRAYSCALE)
fig, ax = plt.subplots(1, 4, figsize=(14, 5))
ax[0].imshow(pencere_al(im, (s.cy, s.cx), GLOBAL_SIZE), cmap='gray')
ax[0].set_title('gurultusuz (val/test)', fontsize=9)
for i in range(1, 4):
    ax[i].imshow(pencere_al(im, (s.cy, s.cx), GLOBAL_SIZE, MAX_CROP_NOISE,
                            MAX_CROP_SIZE_NOISE, rng), cmap='gray')
    ax[i].set_title(f'gurultulu ornek {i} (egitim)', fontsize=9)
for a in ax:
    a.axis('off')
plt.suptitle(f'{s.dosya}  --  {s.classification} / {s.abnormality}')
plt.tight_layout()
plt.savefig(OZET_DIR / 'pencere_augmentasyon.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
ozet = dict(
    goruntu=len(manifest), meme=int(manifest.meme.nunique()), hasta=int(manifest.hasta.nunique()),
    GLOBAL_SIZE=GLOBAL_SIZE, kirpik_h_medyan=float(manifest.h.median()),
    kirpik_w_medyan=float(manifest.w.median()),
    split_hasta={k: len(v) for k, v in split.items()},
    split_meme=manifest.groupby('split').meme.nunique().to_dict(),
    sinif=manifest.drop_duplicates('meme').classification.value_counts().to_dict(),
    eval_vaka=len(eval_cases),
)
for k, v in ozet.items():
    print(f'{k:20s}: {v}')
pd.DataFrame([{k: str(v) for k, v in ozet.items()}]).T.rename(columns={0: 'deger'}) \
    .to_excel(OZET_DIR / 'hazirlik_ozeti.xlsx')
print('\nParca 2 tamam.')